In [7]:
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import data_enrichment
import numpy as np
import importlib

In [8]:
train_df = pd.read_csv('datasets/update_datasets/train_df.csv')
test_df = pd.read_csv('datasets/update_datasets/test_df.csv')

### Finding ESG Factors 
Our objective in this step is to classify each news item into specific ESG factors by leveraging a pretrained model trained on ESG-related texts. The model produces percentage scores for each factor, reflecting the degree of association between the text and the factor. Since a given text may not correspond exclusively to a single factor, using percentages provides a more nuanced representation of its relevance across multiple factors.

In [9]:
train_esg = train_df['news_content'].apply(lambda x: pd.Series(data_enrichment.get_labels_and_scores(x)))
test_esg = test_df['news_content'].apply(lambda x: pd.Series(data_enrichment.get_labels_and_scores(x)))
train_esg.columns = ['finbert_labels', 'finbert_scores']
test_esg.columns = ['finbert_labels', 'finbert_scores']

print(train_esg.head())

                              finbert_labels  \
0  [None, Environmental, Social, Governance]   
1  [None, Environmental, Social, Governance]   
2  [None, Environmental, Social, Governance]   
3  [None, Environmental, Social, Governance]   
4  [None, Environmental, Social, Governance]   

                                      finbert_scores  
0  [0.02972550503909588, 0.010191040113568306, 0....  
1  [0.5855902433395386, 0.010819051414728165, 0.1...  
2  [0.0778643786907196, 0.07067982852458954, 0.80...  
3  [0.035829104483127594, 0.9505242109298706, 0.0...  
4  [0.0602908693253994, 0.045593563467264175, 0.3...  


As shown above, a new column has been added that combines the percentage scores into a single attribute. In the following step, this column will be separated into three distinct attributes, each corresponding to one of the ESG factors.

In [10]:
train_esg[['None', 'Environment', 'Social', 'Governance']] = train_esg.apply(data_enrichment.assign_scores, axis=1)
test_esg[['None', 'Environment', 'Social', 'Governance']] = test_esg.apply(data_enrichment.assign_scores, axis=1)

print(train_esg.head())

                              finbert_labels  \
0  [None, Environmental, Social, Governance]   
1  [None, Environmental, Social, Governance]   
2  [None, Environmental, Social, Governance]   
3  [None, Environmental, Social, Governance]   
4  [None, Environmental, Social, Governance]   

                                      finbert_scores      None  Environment  \
0  [0.02972550503909588, 0.010191040113568306, 0....  0.029726     0.010191   
1  [0.5855902433395386, 0.010819051414728165, 0.1...  0.585590     0.010819   
2  [0.0778643786907196, 0.07067982852458954, 0.80...  0.077864     0.070680   
3  [0.035829104483127594, 0.9505242109298706, 0.0...  0.035829     0.950524   
4  [0.0602908693253994, 0.045593563467264175, 0.3...  0.060291     0.045594   

     Social  Governance  
0  0.749006    0.211077  
1  0.198589    0.205002  
2  0.800480    0.050976  
3  0.009723    0.003924  
4  0.369712    0.524403  


In [13]:
count_nones = train_esg[train_esg['None'] > 0.5].shape[0]

print(count_nones)
print('It can be observed that only a small number of texts have their highest percentage classified as None. For this reason, we will keep them in the dataset.')

43
It can be observed that only a small number of texts have their highest percentage classified as None. For this reason, we will keep them in the dataset.


In [ ]:
train_esg = train_esg.drop(["finbert_labels", "finbert_scores"], axis=1)
test_esg = test_esg.drop(["finbert_labels", "finbert_scores"], axis=1)

In [34]:
train_esg.to_csv("datasets/update_datasets/train_esg.csv", index=False)
test_esg.to_csv("datasets/update_datasets/test_esg.csv", index=False)

#### 768-token classification
The next step focuses on the 768-token classification provided by the pretrained ESG topic model mentioned earlier. In this process, each news item is represented as a vector of 768 features, capturing the semantic information relevant to ESG topics. This high-dimensional representation allows the model to assign probabilities or scores to each ESG factor, effectively quantifying the degree to which the text is associated with each topic. Leveraging these embeddings enables a more precise and nuanced analysis of the news content, facilitating downstream tasks such as classification, clustering, or further predictive modeling


In [15]:
embeddings_matrix = np.vstack(train_df['news_content'].apply(data_enrichment.get_bert_embeddings).values)
emb_cols = [f'feat_{i}' for i in range(768)]
train_esg_embeddings = pd.DataFrame(embeddings_matrix, columns=emb_cols)

embeddings_matrix = np.vstack(test_df['news_content'].apply(data_enrichment.get_bert_embeddings).values)
emb_cols = [f'feat_{i}' for i in range(768)]
test_esg_embeddings = pd.DataFrame(embeddings_matrix, columns=emb_cols)

print(train_esg_embeddings.head())

     feat_0    feat_1    feat_2    feat_3    feat_4    feat_5    feat_6  \
0  1.039257  0.554901 -1.125827  0.939134  0.079323 -0.722375 -0.140655   
1 -1.455047 -0.730238  0.259169  0.382989 -1.059021  0.005821 -0.120152   
2  0.325190 -0.256381 -0.621233  0.365065 -0.442346 -1.062278 -0.118988   
3  0.073834 -0.443442  0.524872 -0.717891  0.042195 -1.052497  0.324664   
4  1.293768  1.495430 -0.599434  2.206621  0.310329 -1.474391 -0.475058   

     feat_7    feat_8    feat_9  ...  feat_758  feat_759  feat_760  feat_761  \
0  1.434603 -1.293951 -0.328192  ...  1.734010  0.849596  1.454530 -0.400194   
1  1.540400 -0.300644 -0.796230  ...  2.052773 -0.564044  0.939710 -0.337704   
2  0.517100 -0.354877  0.876494  ...  0.962362  2.002472  1.104132  0.005756   
3  0.122817 -0.250954  0.772999  ... -0.540775  0.839328 -0.416038  0.760624   
4  2.391380 -0.870735  0.781607  ...  0.099791  2.046669  0.864141  0.099384   

   feat_762  feat_763  feat_764  feat_765  feat_766  feat_767  
0  0

In [ ]:
train_esg_embeddings.to_csv("datasets/update_datasets/train_esg_embeddings.csv", index=False)
test_esg_embeddings.to_csv("datasets/update_datasets/test_esg_embeddings.csv", index=False)

### Sentiment Analysis 
We will apply sentiment analysis to the news_content attribute. The rationale behind this step is that the sentiment of the news, whether positive or negative, can often provide valuable insights into the potential implications of an ESG issue. Identifying sentiment in this way allows us to account not only for the factual content of the news but also for its tone and underlying perspective, which can influence how the issue is perceived and prioritized. Using these sentiment percentages enriches our dataset and provides a more nuanced understanding of the news content.

In [17]:
sentiment_scores_train = train_df['news_content'].apply(data_enrichment.get_sentiment_probs)
sentiment_scores_test = test_df['news_content'].apply(data_enrichment.get_sentiment_probs)

sentiment_scores_train.head() 

0    {'Environmental_Negative': 0.05020667, 'Enviro...
1    {'Environmental_Negative': 0.05151997, 'Enviro...
2    {'Environmental_Negative': 0.05886808, 'Enviro...
3    {'Environmental_Negative': 0.3349111, 'Environ...
4    {'Environmental_Negative': 0.07632709, 'Enviro...
Name: news_content, dtype: object

In [18]:
with pd.option_context('display.max_colwidth', None):
    print(sentiment_scores_train.head()) 

0    {'Environmental_Negative': 0.05020667, 'Environmental_Neutral': 0.03486242, 'Environmental_Positive': 0.108010456, 'Social_Negative': 0.110986464, 'Social_Neutral': 0.13048553, 'Social_Positive': 0.61817575, 'Governance_Negative': 0.059293218, 'Governance_Neutral': 0.030225439, 'Governance_Positive': 0.09322014}
1        {'Environmental_Negative': 0.05151997, 'Environmental_Neutral': 0.040244415, 'Environmental_Positive': 0.1780351, 'Social_Negative': 0.07169157, 'Social_Neutral': 0.09477854, 'Social_Positive': 0.784463, 'Governance_Negative': 0.052783877, 'Governance_Neutral': 0.037587732, 'Governance_Positive': 0.15867086}
2       {'Environmental_Negative': 0.05886808, 'Environmental_Neutral': 0.042912845, 'Environmental_Positive': 0.2884205, 'Social_Negative': 0.061040547, 'Social_Neutral': 0.08128048, 'Social_Positive': 0.8420485, 'Governance_Negative': 0.05248932, 'Governance_Neutral': 0.047302965, 'Governance_Positive': 0.21425015}
3     {'Environmental_Negative': 0.3349111,

In [19]:
sentiment_train = sentiment_scores_train.apply(pd.Series)
sentiment_test = sentiment_scores_test.apply(pd.Series)

In [20]:
column_names= ['Environmental_Negative', 'Environmental_Neutral', 'Environmental_Positive', 'Social_Negative', 'Social_Neutral', 'Social_Positive', 'Governance_Negative', 'Governance_Neutral', 'Governance_Positive']
sentiment_train[column_names] = sentiment_train[column_names].div(sentiment_train[column_names].sum(axis=1), axis=0)
sentiment_test[column_names] = sentiment_test[column_names].div(sentiment_test[column_names].sum(axis=1), axis=0)

In [21]:
sentiment_train = data_enrichment.aggregate_sentiments(sentiment_train)
sentiment_test = data_enrichment.aggregate_sentiments(sentiment_test)
print(sentiment_train.head())

   Environmental_Negative  Environmental_Neutral  Environmental_Positive  \
0                0.040638               0.028218                0.087425   
1                0.035053               0.027381                0.121131   
2                0.034862               0.025413                0.170803   
3                0.218991               0.038459                0.264439   
4                0.037197               0.035131                0.268594   

   Social_Negative  Social_Neutral  Social_Positive  Governance_Negative  \
0         0.089834        0.105616         0.500358             0.047993   
1         0.048777        0.064485         0.533730             0.035913   
2         0.036148        0.048134         0.498663             0.031084   
3         0.091563        0.046871         0.174401             0.083472   
4         0.024329        0.047000         0.423860             0.027957   

   Governance_Neutral  Governance_Positive       pos       neu       neg  
0          

In [22]:
sentiment_train = sentiment_train[['pos', 'neu', 'neg']]
sentiment_test = sentiment_test[['pos', 'neu', 'neg']]

In [23]:
sentiment_train.to_csv("datasets/update_datasets/train_sent.csv", index=False)
sentiment_test.to_csv("datasets/update_datasets/test_sent.csv", index=False)

#### 768-token classification
To enrich the dataset for downstream modeling, two complementary feature sets are incorporated. First, the pretrained ESG topic model provides 768-dimensional embeddings for each news item, capturing semantic information relevant to ESG topics and enabling a nuanced representation of text associations with different factors. Second, sentiment analysis is applied using VADER, producing percentage scores for positive, negative, and neutral sentiments. These sentiment features capture the tone of the news content, offering additional insights into the potential impact and perception of ESG issues. Together, the ESG embeddings and sentiment scores create a comprehensive feature set that supports more accurate and informed predictive modeling.

In [24]:
embeddings = train_df['news_content'].apply(data_enrichment.get_sent_embeddings)
train_sent_embeddings = pd.DataFrame(embeddings.tolist(), index=train_df.index)

embeddings = test_df['news_content'].apply(data_enrichment.get_sent_embeddings)
test_sent_embeddings = pd.DataFrame(embeddings.tolist(), index=test_df.index)

In [25]:
train_sent_embeddings.to_csv("datasets/update_datasets/train_sent_embeddings.csv", index=False)
test_sent_embeddings.to_csv("datasets/update_datasets/test_sent_embeddings.csv", index=False)

### A merged dataframe with both ESG topics and Sentiment Analysis

In [36]:
train_esg_sent = pd.concat([train_esg, sentiment_train], axis=1)
test_esg_sent = pd.concat([test_esg, sentiment_test], axis=1)

In [38]:
train_esg_sent.to_csv("datasets/update_datasets/train_esg_sent.csv", index=False)
test_esg_sent.to_csv("datasets/update_datasets/test_esg_sent.csv", index=False)

#### 768-token classification

In [39]:
train_esg_sent_embeddings = pd.concat([train_esg_embeddings, train_sent_embeddings], axis=1)
test_esg_sent_embeddings = pd.concat([test_esg_embeddings, test_sent_embeddings], axis=1)

In [40]:
train_esg_sent_embeddings.to_csv("datasets/update_datasets/train_esg_sent_embeddings.csv", index=False)
test_esg_sent_embeddings.to_csv("datasets/update_datasets/test_esg_sent_embeddings.csv", index=False)